# Prompt refinement: vague vs. structured

Companion notebook for **lesson 3.1**. The same support emails go through a vague prompt and a refined one (role, task, context, output format). The refined output is checked by a Pydantic model, so "better" is measurable, not a feeling.

**Run it in Colab, setup cell first.** It uses Colab's built-in models (no API key); uncomment a `MODEL = ...` line there to choose one.

In [ ]:
# Setup: Colab's built-in models, no API key needed. Run this cell first.
from google.colab import ai
print("Available models:", ", ".join(ai.list_models()))

MODEL = None                              # Colab's default; uncomment a line below to choose
# MODEL = "google/gemini-2.5-flash"
# MODEL = "google/gemini-2.5-pro"
# MODEL = "google/gemini-3.8-flash"

def get_completion(messages):
    """Send chat messages to the model. Colab's generate_text takes one prompt, so roles are inlined."""
    prompt = "\n\n".join(f"{m['role'].upper()}:\n{m['content']}" for m in messages) + "\n\nASSISTANT:\n"
    return ai.generate_text(prompt, model_name=MODEL) if MODEL else ai.generate_text(prompt)

## 1 · The emails

In [1]:
EMAILS = [
    "Hi, I was charged twice for my last order (Order #8675309). I thought my subscription was paused. "
    "Can you reverse the extra charge? Thanks, Alex",
    "Since this morning the app crashes every time I open the reports page. I've tried reinstalling. "
    "We have a board meeting in two hours.",
    "Hello! Do you offer discounts for non-profit organizations?",
]

## 2 · Two prompt versions

The vague one leaves every decision to the model. The refined one adds a **role**, a clear **task**, **context** (what each category means) and an **output format** whose allowed values a Pydantic model enforces.

In [2]:
import json, re
from typing import Literal
from pydantic import BaseModel, ValidationError

VAGUE = "Please categorize the following email:\n\n{email}"

REFINED = """You are a support agent who categorizes incoming emails for a ticketing system.

Categories:
- Billing: charges, subscriptions, refunds
- Technical Support: product functionality, bugs, outages
- General Inquiry: anything else

Urgency: High = money lost or work blocked now; Medium = needs a reply soon; Low = informational.

Respond with only a JSON object with keys: category (one of the categories), summary (one
sentence), urgency (High, Medium or Low), order_id (the order number if present, otherwise null).

Email:
{email}"""

class Ticket(BaseModel):
    category: Literal["Billing", "Technical Support", "General Inquiry"]
    summary: str
    urgency: Literal["High", "Medium", "Low"]
    order_id: str | None

def parse_ticket(reply: str) -> Ticket | str:
    """The gate: a Ticket, or the reason the reply was rejected."""
    raw = re.sub(r"^\s*`{3}(?:json)?\s*|\s*`{3}\s*$", "", reply.strip())
    try:
        return Ticket.model_validate_json(raw)
    except ValidationError as e:
        return "; ".join(f"{'.'.join(map(str, err['loc'])) or 'reply'}: {err['msg']}" for err in e.errors())

## 3 · The gate on example replies

No model needed: a good reply, a category the system doesn't have, and a correct-but-chatty answer like the vague prompt's.

In [3]:
for reply in [
    '{"category": "Billing", "summary": "Double charge on order 8675309.", "urgency": "High", "order_id": "8675309"}',
    '{"category": "Payments", "summary": "Double charge.", "urgency": "urgent", "order_id": null}',
    "This email appears to be a billing issue related to a double charge on an order.",
]:
    result = parse_ticket(reply)
    print(("✅ " + repr(result)) if isinstance(result, Ticket) else ("❌ " + result))

✅ Ticket(category='Billing', summary='Double charge on order 8675309.', urgency='High', order_id='8675309')
❌ category: Input should be 'Billing', 'Technical Support' or 'General Inquiry'; urgency: Input should be 'High', 'Medium' or 'Low'
❌ reply: Invalid JSON: expected value at line 1 column 1


## 4 · Compare the versions on every email

The vague prompt usually answers correctly, but in a different shape each time. The refined prompt should produce a valid `Ticket` for every email.

In [ ]:
for email in EMAILS:
    vague = get_completion([{"role": "user", "content": VAGUE.format(email=email)}])
    refined = parse_ticket(get_completion([{"role": "user", "content": REFINED.format(email=email)}]))
    print(f"EMAIL   {email[:70]}…")
    print(f"vague   {' '.join(vague.split())[:110]}")
    print(f"refined {'✅ ' + repr(refined) if isinstance(refined, Ticket) else '❌ ' + refined}\n")